# <font color='blue'>Data Science Academy</font>
# <font color='blue'>Deep Learning Para Aplicações de IA com PyTorch e Lightning</font>
## <font color='blue'>Mini-Projeto 8</font>
### <font color='blue'>Fiscalizando o Uso de Recursos Públicos com IA Para Detectar Buracos em Imagens de Estradas</font>

![DSA](images/MP8.png)

## Instalando e Carregando Pacotes

In [ ]:
# Versão da Linguagem Python
from platform import python_version
print('Versão da Linguagem Python Usada Neste Jupyter Notebook:', python_version())

In [ ]:
# Para atualizar um pacote, execute o comando abaixo no terminal ou prompt de comando:
# pip install -U nome_pacote

# Para instalar a versão exata de um pacote, execute o comando abaixo no terminal ou prompt de comando:
# !pip install nome_pacote==versão_desejada

# Depois de instalar ou atualizar o pacote, reinicie o jupyter notebook.

# Instala o pacote watermark. 
# Esse pacote é usado para gravar as versões de outros pacotes usados neste jupyter notebook.
!pip install -q -U watermark

In [ ]:
#!pip install -q opencv-python==4.9.0.80
!pip install -q opencv-python

In [ ]:
#!pip install -q tensorflow==2.15
!pip install -q tensorflow

In [ ]:
%env TF_CPP_MIN_LOG_LEVEL=3

In [ ]:
# Imports
import os
import cv2
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from keras.preprocessing import image

In [ ]:
# Versões dos pacotes usados neste jupyter notebook
%reload_ext watermark
%watermark -a "Data Science Academy" --iversions

## Verificando as Imagens

In [ ]:
# Verificando as imagens
for dirname, _, filenames in os.walk('dataset'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
plt.imshow(cv2.imread("dataset/normal/8.jpg"))

In [ ]:
plt.imshow(cv2.imread("dataset/potholes/10.jpg"))

## Pré-Processamento de Dados

In [ ]:
# Gerador de dados de treino
datagen_treino = tf.keras.preprocessing.image.ImageDataGenerator(rescale = 1./255,
                                                                 shear_range = 0.2,
                                                                 zoom_range = 0.2,
                                                                 horizontal_flip = True,
                                                                 validation_split = 0.2)

In [ ]:
# Dataset de treino
dataset_treino = datagen_treino.flow_from_directory('dataset/',
                                                    target_size = (64, 64),
                                                    batch_size = 32,
                                                    class_mode = 'binary',
                                                    subset = "training")

In [ ]:
# Dataset de validação
dataset_valid = datagen_treino.flow_from_directory("dataset", 
                                                   target_size = (64, 64),
                                                   batch_size = 32,
                                                   class_mode = 'binary',
                                                   subset = 'validation')

## Construção do Modelo

In [ ]:
# Cria sequência de camadas
modelo_dsa = tf.keras.models.Sequential()

In [ ]:
# Camada de convolução
modelo_dsa.add(tf.keras.layers.Conv2D(filters = 32, 
                                      kernel_size = 3, 
                                      activation = 'relu', 
                                      input_shape = [64, 64, 3]))

In [ ]:
# Camada de Pooling
modelo_dsa.add(tf.keras.layers.MaxPool2D(pool_size = 2, 
                                         strides = 2))

In [ ]:
# Segunda camada de convolução e pooling
modelo_dsa.add(tf.keras.layers.Conv2D(filters = 32, kernel_size = 3, activation = 'relu'))
modelo_dsa.add(tf.keras.layers.MaxPool2D(pool_size = 2, strides = 2))

In [ ]:
# Flatten
modelo_dsa.add(tf.keras.layers.Flatten())

In [ ]:
# Camada totalmente conectada
modelo_dsa.add(tf.keras.layers.Dense(units = 128, activation = 'relu'))

In [ ]:
# Camada de saída
modelo_dsa.add(tf.keras.layers.Dense(units = 1, activation = 'sigmoid'))

In [ ]:
# Compilação do modelo
modelo_dsa.compile(optimizer = 'adam', loss = 'binary_crossentropy', metrics = ['accuracy'])

Desafio: Converta este mini-projeto do TensorFlow para o PyTorch.

## Treinamento do Modelo

In [ ]:
modelo_dsa.summary()

In [ ]:
%%time
modelo_dsa.fit(x = dataset_treino, validation_data = dataset_valid, epochs = 25)

## Deploy e Uso do Modelo

In [ ]:
# Nova imagem
nova_imagem = image.load_img('dataset/nova_imagem1.jpg', target_size = (64, 64))
#nova_imagem = image.load_img('dataset/nova_imagem2.jpg', target_size = (64, 64))

In [ ]:
# Converte a imagem em array
nova_imagem = image.img_to_array(nova_imagem)

In [ ]:
# Ajusta as dimensões
nova_imagem = np.expand_dims(nova_imagem, axis = 0)

In [ ]:
# Previsão
previsao = modelo_dsa.predict(nova_imagem)

In [ ]:
previsao

In [ ]:
# Índices
dataset_treino.class_indices

In [ ]:
if previsao[0][0] == 1:
    resultado = 'Buraco na estrada. Comunique o Órgão Responsável no Governo!'
else:
    resultado = 'Imagem Normal!'

In [ ]:
print(resultado)

# Fim